# §5 — Freeze-configuration comparison

Trains four configurations for **1500 steps each** on top of the best injection + masking from the previous problems (`all_patches` + `image_bidir`):

| Cfg | Encoder (ViT) | Projector | Decoder (SmolLM2) |
|-----|---------------|-----------|--------------------|
| A   | frozen        | trained   | frozen             |
| B   | frozen        | trained   | LoRA (rank 8, α=16) on q_proj/v_proj |
| C   | frozen        | trained   | full FT            |
| D   | full FT       | trained   | full FT            |

Reports val exact-match accuracy, trainable parameter count, and peak GPU memory per configuration.

**LoRA for config B:** `basics.lora.apply_lora_to_attention` only targets our `basics.model.Head` instances and doesn't see SmolLM2's HF attention modules. `scripts.train_vlm.apply_lora_to_decoder` (new) walks the decoder, finds every module that exposes `q_proj`/`v_proj` as direct `nn.Linear` attributes, and wraps those with `LoRALinear` — migrating A/B to the base's device + bf16 dtype so the matmul stays in-dtype.

**Compute warning:** Configs C and D do full decoder backprop and are slow (~1.5–2× the A/B steps). Budget roughly 2 hours on L4 (or ~1 hour on A100) for all four runs in sequence. Each run's `metrics.json` syncs to Drive, so the sweep is resumable across kernel sessions.

**Attention impl:** `image_bidir` requires SDPA (FA2 doesn't accept arbitrary 4D masks), so this notebook hard-pins `ATTN_IMPL = 'sdpa'`.

## 1. Install dependencies

In [1]:
%%capture
!pip -q install -U transformers datasets "sentence-transformers<4.0" accelerate tqdm matplotlib pyyaml einops
!pip -q install --force-reinstall --no-deps --no-cache-dir pillow==11.3.0

## 2. Mount Drive and set up the repo path

In [2]:
import os, sys
from pathlib import Path

USE_DRIVE = True
DRIVE_REPO_ROOT = Path('/content/drive/MyDrive/caltech/junior/hw3/')  # edit if needed
LOCAL_REPO_ROOT = Path('/content/hw3')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    REPO_ROOT = DRIVE_REPO_ROOT
else:
    REPO_ROOT = LOCAL_REPO_ROOT

assert REPO_ROOT.exists(), f'Repo root does not exist: {REPO_ROOT}'
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print('cwd =', os.getcwd())

Mounted at /content/drive
cwd = /content/drive/MyDrive/caltech/junior/hw3


In [3]:
import torch, torchvision

print('torch:', torch.__version__, '| torchvision:', torchvision.__version__, '| cuda:', torch.cuda.is_available())
torchvision.ops.nms(torch.zeros(0, 4), torch.zeros(0), 0.5)
print('torchvision ops OK')

torch: 2.10.0+cu128 | torchvision: 0.25.0+cu128 | cuda: True
torchvision ops OK


## 3. CLEVR-mini data sanity

In [4]:
CLEVR_ROOT = REPO_ROOT / 'data' / 'clevr_mini'
if not (CLEVR_ROOT / 'train.jsonl').exists():
    print(f'CLEVR not found at {CLEVR_ROOT} — downloading...')
    !python scripts/download_clevr.py
else:
    print(f'CLEVR ready at {CLEVR_ROOT}')

assert (CLEVR_ROOT / 'train.jsonl').exists()
assert (CLEVR_ROOT / 'val.jsonl').exists()
assert (CLEVR_ROOT / 'images').exists()

CLEVR ready at /content/drive/MyDrive/caltech/junior/hw3/data/clevr_mini


## 4. Configure paths and the §5 freeze-comparison settings

In [5]:
import yaml

CONFIG_PATH = REPO_ROOT / 'configs' / 'vlm_clevr.yaml'
PRETRAINED_VIT = REPO_ROOT / 'runs' / 'clip_eurosat' / 'best.pt'

assert CONFIG_PATH.exists(), f'config missing: {CONFIG_PATH}'
assert PRETRAINED_VIT.exists(), (
    f'CLIP-pretrained checkpoint missing at {PRETRAINED_VIT}. Run §3 first.'
)

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# Best injection + masking from previous problems (writeup §5).
INJECTION = 'all_patches'
MASK_MODE = 'image_bidir'
ATTN_IMPL = 'sdpa'  # image_bidir requires a 4D mask -> SDPA

NUM_STEPS = 1500
BATCH_SIZE = 32
GRAD_ACCUM = 1
LR = 1.0e-4

FREEZE_CONFIGS = ['A', 'B', 'C', 'D']

print(f'injection={INJECTION}  mask={MASK_MODE}  attn={ATTN_IMPL}')
print(f'num_steps={NUM_STEPS}  batch_size={BATCH_SIZE}  lr={LR}')
print(f'sweeping freeze_configs={FREEZE_CONFIGS}')

injection=all_patches  mask=image_bidir  attn=sdpa
num_steps=1500  batch_size=32  lr=0.0001
sweeping freeze_configs=['A', 'B', 'C', 'D']


## 5. Train each freeze configuration

Sequential loop over A → B → C → D. Each run writes `metrics.json` to `/content/runs/vlm_all_patches_image_bidir_<cfg>/` and single-shot syncs it to Drive. Resumable via `OVERWRITE = False`.

In [6]:
import argparse, json, shutil, time

from scripts.train_vlm import train, _default_run_dir

OVERWRITE = False

LOCAL_RUNS = Path('/content/runs')
LOCAL_RUNS.mkdir(parents=True, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

freeze_metrics = {}
for freeze_cfg in FREEZE_CONFIGS:
    local_dir = _default_run_dir(LOCAL_RUNS, INJECTION, MASK_MODE, freeze_cfg)
    drive_dir = REPO_ROOT / 'runs' / local_dir.name
    drive_metrics_path = drive_dir / 'metrics.json'

    if not OVERWRITE and drive_metrics_path.exists():
        freeze_metrics[freeze_cfg] = json.loads(drive_metrics_path.read_text())
        print(f'[{freeze_cfg}] reusing cached metrics  '
              f'(val_acc={freeze_metrics[freeze_cfg]["final_val_acc"]:.4f}, '
              f'trainable={freeze_metrics[freeze_cfg]["trainable_params"]:,})')
        continue

    local_dir.mkdir(parents=True, exist_ok=True)
    args = argparse.Namespace(
        config=CONFIG_PATH,
        pretrained_vit=PRETRAINED_VIT,
        clevr_root=CLEVR_ROOT,
        injection=INJECTION,
        mask_mode=MASK_MODE,
        freeze_config=freeze_cfg,
        num_steps=NUM_STEPS,
        batch_size=BATCH_SIZE,
        grad_accum=GRAD_ACCUM,
        lr=LR,
        max_len=128,
        attn_impl=ATTN_IMPL,
        output_dir=local_dir,
        device=device,
        summarize=False,
        runs_root=LOCAL_RUNS,
    )
    print(f'\n========== freeze_config={freeze_cfg} ==========')
    t0 = time.time()
    m = train(args, cfg)
    print(f'[{freeze_cfg}] wall = {time.time() - t0:.1f}s  '
          f'val_acc={m["final_val_acc"]:.4f}  '
          f'trainable={m["trainable_params"]:,}  '
          f'peak={m["peak_mem_mb"]:.1f}MB')
    freeze_metrics[freeze_cfg] = m

    drive_dir.mkdir(parents=True, exist_ok=True)
    # Always sync metrics.json. Skip projector.pt for C / D since the trainable
    # checkpoints would be huge (full decoder); keep them local only.
    shutil.copy2(local_dir / 'metrics.json', drive_dir / 'metrics.json')
    if freeze_cfg in ('A', 'B'):
        for f in local_dir.glob('projector.pt'):
            shutil.copy2(f, drive_dir / f.name)
    print(f'synced metrics to {drive_dir}')


========== freeze_config=A ==========


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...


step 200: val_overall=0.0620  loss=0.5925

step 400: val_overall=0.0180  loss=0.5667

step 600: val_overall=0.1800  loss=0.4475

step 800: val_overall=0.0900  loss=0.3843

step 1000: val_overall=0.0980  loss=0.5068

step 1200: val_overall=0.1460  loss=0.5094

step 1400: val_overall=0.1420  loss=0.4643

step 1500: val_overall=0.1360  loss=0.4732
saved /content/runs/vlm_all_patches_image_bidir_B/metrics.json
[B] wall = 326.2s  val_acc=0.1360  trainable=2,886,080  peak=6944.9MB
synced metrics to /content/drive/MyDrive/caltech/junior/hw3/runs/vlm_all_patches_image_bidir_B

========== freeze_config=C ==========
[all_patches/image_bidir/C] trainable=363,888,000  total=374,625,792


[all_patches]:   0%|          | 0/1500 [00:00<?, ?it/s]


step 200: val_overall=0.0640  loss=0.5304

step 400: val_overall=0.1360  loss=0.4308

step 600: val_overall=0.0240  loss=0.4015

step 800: val_overall=0.0460  loss=0.5395

step 1000: val_overall=0.0360  loss=0.4573

step 1200: val_overall=0.0460  loss=0.4299

step 1400: val_overall=0.0500  loss=0.4148

step 1500: val_overall=0.0480  loss=0.4631
saved /content/runs/vlm_all_patches_image_bidir_C/metrics.json
[C] wall = 408.6s  val_acc=0.0480  trainable=363,888,000  peak=9580.0MB
synced metrics to /content/drive/MyDrive/caltech/junior/hw3/runs/vlm_all_patches_image_bidir_C

========== freeze_config=D ==========
[all_patches/image_bidir/D] trainable=374,625,792  total=374,625,792


[all_patches]:   0%|          | 0/1500 [00:00<?, ?it/s]


step 200: val_overall=0.0460  loss=0.5431

step 400: val_overall=0.2900  loss=0.5151

step 600: val_overall=0.1940  loss=0.4389

step 800: val_overall=0.2980  loss=0.4677

step 1000: val_overall=0.2620  loss=0.4801

step 1200: val_overall=0.2620  loss=0.4876

step 1400: val_overall=0.2300  loss=0.3588

step 1500: val_overall=0.2340  loss=0.4008
saved /content/runs/vlm_all_patches_image_bidir_D/metrics.json
[D] wall = 454.1s  val_acc=0.2340  trainable=374,625,792  peak=10009.1MB
synced metrics to /content/drive/MyDrive/caltech/junior/hw3/runs/vlm_all_patches_image_bidir_D


## 6. 4-row deliverable table

In [7]:
cols = [
    ('freeze_cfg',  None,             12),
    ('val_acc',     'final_val_acc',   9),
    ('trainable',   'trainable_params',16),
    ('total',       'total_params',   16),
    ('peak_mem_MB', 'peak_mem_mb',    13),
    ('sec_per_step','sec_per_step',   13),
]
header = ' | '.join(f'{n:>{w}}' for n, _, w in cols)
print()
print(header)
print('-' * len(header))
for cfg in ('A', 'B', 'C', 'D'):
    m = freeze_metrics.get(cfg)
    if m is None:
        continue
    row = []
    for name, key, w in cols:
        v = cfg if key is None else m[key]
        if name == 'val_acc':         s = f'{v:.4f}'
        elif name in ('trainable', 'total'): s = f'{v:,}'
        elif name == 'peak_mem_MB':   s = f'{v:.1f}'
        elif name == 'sec_per_step':  s = f'{v:.3f}'
        else:                         s = str(v)
        row.append(f'{s:>{w}}')
    print(' | '.join(row))


  freeze_cfg |   val_acc |        trainable |            total |   peak_mem_MB |  sec_per_step
----------------------------------------------------------------------------------------------
           A |    0.1400 |        2,066,880 |      374,625,792 |        6752.9 |         0.567
           B |    0.1360 |        2,886,080 |      375,444,992 |        6944.9 |         0.213
           C |    0.0480 |      363,888,000 |      374,625,792 |        9580.0 |         0.258
           D |    0.2340 |      374,625,792 |      374,625,792 |       10009.1 |         0.289


## 7. Writeup notes (5–6 sentences)

Fill in using your numbers. Cover:

- **(1) Trainable parameters scale.** A: ~projector (~2M). B: +LoRA on q_proj/v_proj of every decoder layer (~hundreds of k extra). C: +full 360M decoder. D: +ViT (~11M) on top of C. Cite your actual counts; the ratio C/A is ~150× and is the headline of the trade-off.

- **(2) Peak memory.** Peak GPU memory tracks *trainable* params (because frozen layers don't store grads or AdamW state). A and B should be close; C and D much higher. With AdamW you store 2 fp32 moments per trainable param — full-decoder FT therefore costs roughly `360M × 8 bytes ≈ 2.9 GB` of optimizer state alone, before activations.

- **(3) Accuracy ranking.** On CLEVR-mini at only 1500 steps, expect roughly D ≥ C ≳ B > A. The projector-only ceiling is low (the frozen decoder has no signal saying "these embeddings encode CLEVR objects"); LoRA injects a small amount of task-specific capacity into the decoder; full FT gives the most expressivity but is harder to keep stable. ViT FT (D) sometimes hurts at this step count because the visual features were already aligned via §3 CLIP.

- **(4) Best trade-off.** **B (projector + decoder LoRA)** is usually the Pareto winner: it adds ~2 orders of magnitude fewer trainable params than C, uses near-A memory, and matches most of C's accuracy. This mirrors the §4 conclusion on RESISC.

- **(5) Connection to the two-stage recipe (pretraining → instruction tuning).** Production VLMs follow exactly the pattern of A → B/C: first a *projector-alignment* pretraining phase (only the projector trains, on millions of image–text pairs, so the decoder's text manifold absorbs the new visual modality without touching its language prior) and then an *instruction-tuning* phase (LoRA-style or selective FT on a curated dataset, to actually teach the model task-specific behavior). Our table is a 1500-step microcosm of that story: A on its own is necessary but insufficient — once the projector has learned to talk to the decoder, *some* decoder-side adaptation (B's LoRA) is what closes the rest of the gap, while full FT (C/D) is expensive overkill for small datasets.

Optional: if D scored worse than C, mention that the §3-pretrained ViT was already aligned to the decoder's text features via CLIP, so re-training it from CLEVR signal alone tends to drift it away from a useful representation in only 1500 steps — the same "don't waste a good prior" argument that motivates LoRA over full FT.